In [1]:
import json
import jsonlines
import pandas as pd
from pathlib import Path
from openai import OpenAI
from pydantic import BaseModel
from tqdm.notebook import tqdm

MODEL_NAME = 'gpt-5.4-nano'
REASONING_EFFORT = 'medium'
SERVICE_TIER = 'flex'

DATA_PATH_COMMENTS = '../data/cleaned/cleaned_comments_info.csv'
DATA_PATH_VIDEOS = '../data/cleaned/cleaned_videos_info.csv'
JSON_PATH = f'../data/results/binary/question_detection_{MODEL_NAME.replace(".", "-")}_{REASONING_EFFORT}_{SERVICE_TIER}.jsonl'
OUTPUT_PATH = f'../data/results/binary/question_detection_{MODEL_NAME.replace(".", "-")}_{REASONING_EFFORT}_{SERVICE_TIER}.csv'
USAGE_PATH = f'../data/results/usage/binary/question_detection_{MODEL_NAME.replace(".", "-")}_{REASONING_EFFORT}_{SERVICE_TIER}_usage.csv'

# Registra os tokens usados em cada chamada da API.
LOG_USAGE = True

client = OpenAI()

In [2]:
class QuestionDetection(BaseModel):
    tem_pergunta: bool

In [3]:
DETECTION_PROMPT = """
Determine se o campo "comentario" expressa alguma pergunta ou solicitação de informação, explícita ou implícita.

Você receberá:
- "titulo": título do vídeo em que o comentário foi publicado;
- "comentario_pai": comentário ao qual o comentário atual está respondendo, podendo estar vazio;
- "comentario": comentário que deve ser classificado.

Use "titulo" e "comentario_pai" apenas para compreender o contexto. Não classifique como positiva uma pergunta que esteja somente nesses campos.

Relatos, opiniões, elogios e preocupações sem uma dúvida não contam como pergunta.

Em caso de ambiguidade, classifique como true.

Retorne apenas um objeto JSON válido, sem explicações ou texto adicional:

{"tem_pergunta": true}

ou:

{"tem_pergunta": false}
"""

In [4]:
df_videos = pd.read_csv(DATA_PATH_VIDEOS, usecols=['video_id', 'title'])
df_comments = pd.read_csv(DATA_PATH_COMMENTS)
comments_ids = df_comments['comment_id'].tolist()

df_pais = df_comments[['comment_id', 'comment']].rename(
    columns={'comment_id': 'parent_id', 'comment': 'comentario_pai'}
)

df_completo = df_comments[['comment_id', 'video_id', 'comment', 'parent_id']].merge(
    df_pais,
    on='parent_id',
    how='left'
)

df_final = df_completo.merge(
    df_videos,
    on='video_id',
    how='left'
)

df_final.shape

(225616, 6)

In [5]:
def classify_comments(
    comment_id: str,
    titulo: str,
    comentario_pai: str,
    comentario: str
) -> QuestionDetection:

    user_content = json.dumps({
        "titulo": titulo,
        "comentario_pai": comentario_pai,
        "comentario": comentario
    }, ensure_ascii=False)

    response = client.responses.parse(
        model=MODEL_NAME,
        instructions=DETECTION_PROMPT,
        input=user_content,
        text_format=QuestionDetection,
        reasoning={"effort": REASONING_EFFORT},
        service_tier=SERVICE_TIER
    )

    if LOG_USAGE:
        usage = response.usage
        if usage is None:
            raise ValueError(f"A API não retornou métricas de uso para {comment_id}")

        input_details = usage.input_tokens_details
        output_details = usage.output_tokens_details
        usage_row = pd.DataFrame([{
            "comment_id": comment_id,
            "model": MODEL_NAME,
            "input_tokens": usage.input_tokens,
            "cached_input_tokens": input_details.cached_tokens if input_details else 0,
            "output_tokens": usage.output_tokens,
            "reasoning_tokens": output_details.reasoning_tokens if output_details else 0,
            "total_tokens": usage.total_tokens
        }])
        usage_path = Path(USAGE_PATH)
        usage_path.parent.mkdir(parents=True, exist_ok=True)
        usage_row.to_csv(
            usage_path,
            mode='a',
            header=not usage_path.exists(),
            index=False
        )


    if response.output_parsed is None:
        raise ValueError(
            f"Resposta estruturada ausente para o comentário {comment_id}"
        )

    return response.output_parsed

In [6]:
def extract_and_save(comment_id: str, comentario: str, titulo: str, comentario_pai: str, output_path: str=JSON_PATH):
    result = classify_comments(
        comment_id=comment_id,
        titulo=titulo,
        comentario_pai=comentario_pai,
        comentario=comentario
    )

    output = {
        'comment_id': comment_id,
        'comentario': comentario,
        'titulo': titulo,
        'comentario_pai': comentario_pai,
        'tem_pergunta': result.tem_pergunta
    }

    with jsonlines.open(output_path, 'a') as writer:
        writer.write(output)

In [7]:
processed = set()
try:
    with jsonlines.open(JSON_PATH, 'r') as reader:
        processed = {row["comment_id"] for row in reader}
except FileNotFoundError:
    pass

print(len(processed))

231483


In [8]:
for _, row in tqdm(df_final.iterrows(), total=df_final.shape[0], desc="Processing..."):
    comment_id = row['comment_id']
    comentario = "" if pd.isna(row['comment']) else str(row['comment'])
    titulo = "" if pd.isna(row['title']) else str(row['title'])
    comentario_pai = "" if pd.isna(row['comentario_pai']) else str(row['comentario_pai'])

    if comment_id in processed:
        continue

    extract_and_save(
        comment_id=comment_id,
        comentario=comentario,
        titulo=titulo,
        comentario_pai=comentario_pai
    )

Processing...:   0%|          | 0/225616 [00:00<?, ?it/s]

In [ ]:
df_results = pd.read_json(JSON_PATH, lines=True)
df_results = df_results[df_results['comment_id'].isin(comments_ids)]
df_results.to_csv(OUTPUT_PATH, index=False)

# numero de comentários contendo perguntas (relevantes e irrelevantes)
df_results[df_results['tem_pergunta'] == True].shape

(35479, 5)